In [32]:
# Baseline model class

import pandas as pd
import os
import joblib
import pickle
import json
from sklearn import model_selection, preprocessing, metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize_scalar
from typing import Dict

class TFIDFBaselineModel:
    def __init__(self, training_set, validation_set, text_column: str, label_column: str):
        self.training_set = training_set
        self.validation_set = validation_set
        self.text_column = text_column
        self.label_column = label_column
        self.vectorizer = TfidfVectorizer()
        self.label_encoder = LabelEncoder()
        self.model = LogisticRegression(max_iter=1000)

    def preprocess_data(self):
        # Fit the TF-IDF vectorizer on the training data and transform both training and validation data

        self.X_train = self.vectorizer.fit_transform(self.training_set[self.text_column])
        self.X_validation = self.vectorizer.transform(self.validation_set[self.text_column])

        # Encode the labels
        self.y_train = self.label_encoder.fit_transform(self.training_set[self.label_column])
        self.y_validation = self.label_encoder.transform(self.validation_set[self.label_column])

    def train_model(self):
        # Train the logistic regression model
        self.model.fit(self.X_train, self.y_train)

    def evaluate_model(self):
        # Make predictions on the validation set
        y_pred = self.model.predict(self.X_validation)

        # Calculate accuracy
        accuracy = metrics.accuracy_score(self.y_validation, y_pred)
        classification_report = metrics.classification_report(self.y_validation, y_pred, target_names=self.label_encoder.classes_)
        brier = metrics.brier_score_loss(self.y_validation, self.model.predict_proba(self.X_validation)[:, 1])
        print(f'Validation Accuracy: {accuracy:.4f}')
        print(f'Classification Report: \n{classification_report}')
        print(f'Validation Brier score: {brier:.4f}')

    def run_pipeline(self):
        self.preprocess_data()
        self.train_model()
        self.evaluate_model()

    def save_model(self, output_dir):
        # Save the trained model, vectorizer, and label encoder to the specified output directory
        os.makedirs(output_dir, exist_ok=True)
        artifacts = {
                    'model': self.model,
                    'vectorizer': self.vectorizer,
                    'encoder': self.label_encoder
                    }
                
        filepath = os.path.join(output_dir, 'uncalibrated_meta.pkl')
        with open(filepath, 'wb') as f:
            pickle.dump(artifacts, f)
        print(f'Uncalibrated model artifacts saved to {filepath}')

    def save_metrics(self, output_dir, output_format='txt'):
        # Save the evaluation metrics to a text file in the specified output directory
        os.makedirs(output_dir, exist_ok=True)
        y_pred = self.model.predict(self.X_validation)
        accuracy = metrics.accuracy_score(self.y_validation, y_pred)
        classification_report = metrics.classification_report(self.y_validation, y_pred, target_names=self.label_encoder.classes_)
        if output_format == 'txt':
            with open(os.path.join(output_dir, 'metrics.txt'), 'w') as f:
                f.write(f'Validation Accuracy: {accuracy:.4f}\n')
                f.write(f'Classification Report: \n{classification_report}\n')
        elif output_format == 'json':
            with open(os.path.join(output_dir, 'metrics.json'), 'w') as f:
                json.dump({
                    'validation_accuracy': accuracy,
                    'classification_report': classification_report
                }, f)


        

In [30]:
training = pd.read_csv('../../data/splits/training_50_50.csv')
validation = pd.read_csv('../../data/splits/validation_50_50.csv')

baseline_model = TFIDFBaselineModel(training, validation, text_column='Full_Text', label_column='Label')

baseline_model.run_pipeline()


Validation Accuracy: 0.9573
Classification Report: 
              precision    recall  f1-score   support

      Benign       0.95      0.96      0.96      2000
   Malicious       0.96      0.95      0.96      2000

    accuracy                           0.96      4000
   macro avg       0.96      0.96      0.96      4000
weighted avg       0.96      0.96      0.96      4000

Validation Brier score: 0.0421


In [10]:
baseline_model.save_metrics(output_dir='../../experiments/binary/baseline/uncalibrated', output_format='txt')

In [33]:
baseline_model.save_model(output_dir='../../models/binary/baseline/uncalibrated')

Uncalibrated model artifacts saved to ../../models/binary/baseline/uncalibrated\uncalibrated_meta.pkl
Model, vectorizer, and label encoder saved to ../../models/binary/baseline/uncalibrated


In [34]:
import os
import joblib
import pickle
import numpy as np
from scipy.optimize import minimize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV

# A class for calibrating the baseline model using temperature scaling

class BaselineCalibrator:
    def __init__(self, model, vectorizer, encoder, calibration_set, validation_set, text_column: str, label_column: str, temperature: float = 1.0):
        self.original_model = model
        self.calibration_set = calibration_set
        self.validation_set = validation_set
        self.text_column = text_column
        self.label_column = label_column
        self.vectorizer = vectorizer
        self.label_encoder = encoder
        self.temperature = temperature

    def preprocess_data(self):
        # Fit the TF-IDF vectorizer on the calibration data and transform both calibration and validation data
        self.X_calibration = self.vectorizer.transform(self.calibration_set[self.text_column])
        self.X_validation = self.vectorizer.transform(self.validation_set[self.text_column])

        # Encode the labels
        self.y_calibration = self.label_encoder.transform(self.calibration_set[self.label_column])
        self.y_validation = self.label_encoder.transform(self.validation_set[self.label_column])

    def _get_logits(self, X):
        # Extract raw logits (before sigmoid/softmax conversion) from LogisticRegression
        # decision_function returns: w * X + b
        return self.original_model.decision_function(X)

    def calibrate(self):
        # Optimize the temperature parameter to minimize the loss
        raw_logits = self._get_logits(self.X_calibration)

        def objective(T):
            # Prevent division by zero
            T = max(T[0], 1e-5) 
            
            # Apply temperature scaling to logits
            scaled_logits = raw_logits / T
            
            # Convert scaled logits to probabilities using sigmoid
            probs = 1 / (1 + np.exp(-scaled_logits))
            
            # Return log loss against true labels
            return log_loss(self.y_calibration, probs)

        # 3. Optimize the temperature parameter starting at T=1.0
        result = minimize(objective, x0=[1.0], method='Nelder-Mead')
        self.temperature = max(result.x[0], 1e-5)
        print(f"Optimized Temperature: {self.temperature:.4f}")

    def predict_proba(self, X):
        # Predict probabilities using the calibrated model
        raw_logits = self._get_logits(X)
        scaled_logits = raw_logits / self.temperature
        probs = 1 / (1 + np.exp(-scaled_logits))
        return np.vstack([1 - probs, probs]).T  # Return as a 2D array with shape (n_samples, 2)

    def evaluate_calibration(self):
        # Evaluate the calibrated model on the validation set
        scaled_logits = self.predict_proba(self.X_validation)
        loss = log_loss(self.y_validation, scaled_logits)
        brier = brier_score_loss(self.y_validation, scaled_logits[:, 1])
        accuracy = np.mean(np.argmax(scaled_logits, axis=1) == self.y_validation)
        classification_report = metrics.classification_report(self.y_validation, np.argmax(scaled_logits, axis=1), target_names=self.label_encoder.classes_)

        print(f'Validation Loss after calibration: {loss:.4f}')
        print(f'Validation Brier score after calibration: {brier:.4f}')
        print(f'Validation Accuracy after calibration: {accuracy:.4f}')
        print(f'Classification Report after calibration: \n{classification_report}')

    def run_calibration_pipeline(self):
        self.preprocess_data()
        self.calibrate()
        self.evaluate_calibration()

    def save_calibrated_model(self, output_dir):
        # Save both the original model structure and the optimal temperature setting
        artifacts = {
            'model': self.original_model,
            'temperature': self.temperature,
            'vectorizer': self.vectorizer,
            'encoder': self.label_encoder
        }
        
        filepath = os.path.join(output_dir, 'temperature_calibrated_meta.pkl')
        with open(filepath, 'wb') as f:
            pickle.dump(artifacts, f)
        print(f'Calibrated model artifacts saved to {filepath}')
        


    

In [37]:
model_artifacts_path = '../../models/binary/baseline/uncalibrated/uncalibrated_meta.pkl'

model_data = pickle.load(open(model_artifacts_path, 'rb'))

model = model_data['model']
vectorizer = model_data['vectorizer']
encoder = model_data['encoder']

calibration_set = pd.read_csv('../../data/splits/calibration_80_20.csv')
validation_set = pd.read_csv('../../data/splits/validation_80_20.csv')

calibrator = BaselineCalibrator(model=model, vectorizer=vectorizer, encoder=encoder, calibration_set=calibration_set, validation_set=validation_set, text_column='Full_Text', label_column='Label')

calibrator.run_calibration_pipeline()

Optimized Temperature: 0.4121
Validation Loss after calibration: 0.1059
Validation Brier score after calibration: 0.0277
Validation Accuracy after calibration: 0.9675
Classification Report after calibration: 
              precision    recall  f1-score   support

      Benign       0.99      0.97      0.98      3200
   Malicious       0.89      0.96      0.92       800

    accuracy                           0.97      4000
   macro avg       0.94      0.97      0.95      4000
weighted avg       0.97      0.97      0.97      4000



In [28]:
calibrator.save_calibrated_model(output_dir='../../models/binary/baseline/calibrated')

Calibrated model artifacts saved to ../../models/binary/baseline/calibrated\temperature_calibrated_meta.pkl


In [ ]:
# BERT Model Class and TextDataset Class

import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import os
import pickle

class TextDataset(Dataset):

    def __init__(self, dataset, mode, max_len, encoder=None):
        self.texts = dataset['Full_Text'].values
        self.labels = dataset['Label'].values
        self.tokenizer = AutoTokenizer.from_pretrained(mode)
        self.max_len = max_len
        self.encoder = encoder if encoder is not None else LabelEncoder()

    def preprocess_labels(self, training=True):
        if training:
            self.labels = self.encoder.fit_transform(self.labels)
        else:
            self.labels = self.encoder.transform(self.labels)
        return self.encoder

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
    }

    def save_encoder(self, output_dir):
        # Save the fitted label encoder to the specified output directory
        os.makedirs(output_dir, exist_ok=True)
        encoder_path = os.path.join(output_dir, 'label_encoder.pkl')
        with open(encoder_path, 'wb') as f:
            pickle.dump(self.encoder, f)
        print(f'Label encoder saved to {encoder_path}')


import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn import metrics

class BERTClassifier(nn.Module):
    def __init__(self, n_classes, train_loader, val_loader, optimizer, criterion, epochs, learning_rate, pretrained_model_name='answerdotai/ModernBERT-base'):
        super(BERTClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(pretrained_model_name)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.optimizer = optimizer
        self.criterion = criterion
        self.epochs = epochs
        self.learning_rate = learning_rate

        self.training_losses = []
        self.validation_losses = []
        self.all_preds = []
        self.all_labels = []

        # metrics for evaluation
        self.accuracy = None
        self.classification_report = None
        self.confusion_matrix = None
        self.loss_curve = None

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        last_hidden = outputs.last_hidden_state 
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_embeddings = torch.sum(last_hidden * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        pooled_output = sum_embeddings / sum_mask
        
        output = self.drop(pooled_output)
        return self.out(output)
    
    def train_model(self, device):
        if self.optimizer == "adamw":
            optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        else:
            optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)

        criterion = self.criterion
        epochs = self.epochs

        # Determine device type string safely
        dev_type = 'cuda' if 'cuda' in str(device) else 'cpu'
        # CPU autocast uses bfloat16, GPU uses float16
        amp_dtype = torch.float16 if dev_type == 'cuda' else torch.bfloat16

        scaler = torch.amp.GradScaler(device=dev_type) if dev_type == 'cuda' else None

        if self.training_losses is not None and self.validation_losses is not None:
            self.training_losses = [] # Reset loss lists at the start of training
            self.validation_losses = []
        
        for epoch in range(epochs):
            self.train()
            total_loss = 0
            for batch in self.train_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                optimizer.zero_grad()

                with torch.amp.autocast(device_type=dev_type, dtype=amp_dtype):
                    outputs = self(input_ids, attention_mask)
                    loss = criterion(outputs, labels)

                # 4. Scale loss and step using the scaler
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

                total_loss += loss.item()

            self.training_losses.append(total_loss / len(self.train_loader))

            self.eval()
            validation_loss = 0
            with torch.no_grad():
                for batch in self.val_loader:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = batch['labels'].to(device)

                    outputs = self(input_ids, attention_mask)
                    loss = criterion(outputs, labels)
                    validation_loss += loss.item() 

            self.validation_losses.append(validation_loss / len(self.val_loader))

            print(f'Epoch {epoch + 1}/{epochs}, Training Loss: {total_loss / len(self.train_loader)}, Validation Loss: {validation_loss / len(self.val_loader)}')
            
        print(f'Training Loss: {total_loss / len(self.train_loader)}, Validation Loss: {validation_loss / len(self.val_loader)}')

    def evaluate_model(self, device):
        self.eval()
        self.all_preds = []
        self.all_labels = []

        for batch in self.val_loader:
            with torch.no_grad():
                outputs = self(batch['input_ids'].to(device), batch['attention_mask'].to(device))

            preds = torch.argmax(outputs, dim=1)
            self.all_preds.extend(preds.cpu().numpy())
            self.all_labels.extend(batch['labels'].cpu().numpy())
                
        self.accuracy = metrics.accuracy_score(self.all_labels, self.all_preds)
        self.classification_report = classification_report(self.all_labels, self.all_preds, target_names=["Benign", "Malicious"])
        
        print(f'Validation Accuracy: {self.accuracy}')
        print(f'Classification Report:\n{self.classification_report}')

    def run_pipeline(self, device):
        self.to(device)
        self.train_model(device)
        self.evaluate_model(device)
        self.plot_visuals()

    def plot_visuals(self):
        assert hasattr(self, 'all_labels'), "Please run evaluate_model() before plotting visuals."
        cm = confusion_matrix(self.all_labels, self.all_preds)
        class_names = getattr(self.val_loader.dataset, 'classes', None)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
        fig_cm, ax = plt.subplots(figsize=(6, 6))
        disp.plot(cmap=plt.cm.Blues, ax=ax)
        self.confusion_matrix = fig_cm

        loss_curve = plt.figure()
        plt.plot(self.training_losses, label='Training Loss')
        plt.plot(self.validation_losses, label='Validation Loss')
        plt.title('Loss Curve')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        self.loss_curve = loss_curve

    def save_model(self, path):
        if os.path.dirname(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save(self.state_dict(), path)

    def save_metrics(self, path, output_format='txt'):

        os.makedirs(os.path.dirname(path), exist_ok=True)
        if output_format == 'txt':
            with open(path, 'w') as f:
                f.write(f'Validation Accuracy: {self.accuracy}\n')
                f.write(f'Classification Report:\n{self.classification_report}\n')

        elif output_format == 'json':
            with open(path, 'w') as f:
                json.dump({
                    'validation_accuracy': self.accuracy,
                    'classification_report': self.classification_report
                }, f, indent=4)
        else:
            raise ValueError(f"Unsupported output format: {output_format}")

        assert hasattr(self, 'confusion_matrix') and hasattr(self, 'loss_curve'), "Please run plot_visuals() before saving the visuals."
        
        output_dir = os.path.dirname(path) if os.path.dirname(path) else '.'
        self.confusion_matrix.savefig(os.path.join(output_dir, 'confusion_matrix.png'), bbox_inches='tight')
        self.loss_curve.savefig(os.path.join(output_dir, 'loss_curve.png'), bbox_inches='tight')

        plt.close(self.confusion_matrix)
        plt.close(self.loss_curve)

        

In [2]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

In [3]:
import pandas as pd

training_data = pd.read_csv('../../data/splits/training_50_50.csv')
validation_data = pd.read_csv('../../data/splits/validation_50_50.csv')

training_dataset = TextDataset(training_data, mode='answerdotai/ModernBERT-base', max_len=512)
fitted_encoder = training_dataset.preprocess_labels(training=True)

validation_dataset = TextDataset(validation_data, mode='answerdotai/ModernBERT-base', max_len=512, encoder=fitted_encoder)
validation_dataset.preprocess_labels(training=False)

train_loader = DataLoader(training_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(validation_dataset, batch_size=16, shuffle=False, num_workers=0)

In [ ]:
epochs = 5
optimizer = "adamw"
learning_rate = 2e-5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = BERTClassifier(
        n_classes=2,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=nn.CrossEntropyLoss(),
        epochs=epochs,
        learning_rate=learning_rate,
        pretrained_model_name='answerdotai/ModernBERT-base'
    )
bert_model.run_pipeline(device=device)
# bert_model.save_model(path='models/binary/bert/uncalibrated')
# bert_model.save_metrics(path='experiments/binary/bert/uncalibrated', output_format='txt')

In [ ]:
# Temperature Scaling Class for Calibration Set

class TemperatureScaling(nn.Module):
    def __init__(self):
        super(TemperatureScaling, self).__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    